<h2>Description</h2>

Dans ce code, nous allons établir un modèle afin de prédire le débit horaire sur la rue des saints-peres.

Imports

In [7]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance

doc = 'sts_peres.csv'

df_final = pd.read_csv('../datasets_axes_with_all_features/'+ doc, sep=';')

In [8]:
df_final.describe()

,Unnamed: 0,Identifiant arc,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval,heure,dow,mois,annee,...,duree prec (en min),force moyenne vent (m/s),Température,ensoleillement (en min),est_pieton,est_vacances,est_avant_vacances,est_ferie,est_avant_ferie,est_weekend
count,8723.000000,8723.0,1380.000000,1380.000000,8723.0,8723.0,8723.000000,8723.000000,8723.000000,8723.000000,...,8558.000000,8558.000000,8558.000000,8558.000000,8723.000000,8723.000000,8723.000000,8723.000000,8723.000000,8723.000000
mean,4361.000000,191.0,469.554348,7.147420,114.0,119.0,11.506592,2.994612,6.656082,2024.803508,...,3.912713,2.932344,13.242487,13.199112,0.012610,0.357560,0.016623,0.030265,0.030265,0.283159
std,2518.257533,0.0,224.988381,4.277270,0.0,0.0,6.919745,1.996062,3.431450,0.397368,...,12.817157,1.319631,6.740669,22.053415,0.111592,0.479309,0.127860,0.171325,0.171325,0.450559
min,0.000000,191.0,34.000000,0.242220,114.0,119.0,0.000000,0.000000,1.000000,2024.000000,...,0.000000,0.000000,-3.600000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2180.500000,191.0,265.500000,3.434720,114.0,119.0,6.000000,1.000000,4.000000,2025.000000,...,0.000000,2.000000,8.600000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,4361.000000,191.0,523.500000,6.982220,114.0,119.0,12.000000,3.000000,7.000000,2025.000000,...,0.000000,2.700000,13.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,6541.500000,191.0,651.000000,10.014165,114.0,119.0,17.000000,5.000000,10.000000,2025.000000,...,0.000000,3.700000,17.800000,21.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,8722.000000,191.0,1083.000000,26.115560,114.0,119.0,23.000000,6.000000,12.000000,2025.000000,...,60.000000,9.300000,37.600000,60.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [9]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8723 entries, 0 to 8722
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Unnamed: 0                 8723 non-null   int64  
 1   Identifiant arc            8723 non-null   int64  
 2   Libelle                    8723 non-null   object 
 3   Date et heure de comptage  8723 non-null   object 
 4   Débit horaire              1380 non-null   float64
 5   Taux d'occupation          1380 non-null   float64
 6   Etat trafic                8723 non-null   object 
 7   Identifiant noeud amont    8723 non-null   int64  
 8   Libelle noeud amont        8723 non-null   object 
 9   Identifiant noeud aval     8723 non-null   int64  
 10  Libelle noeud aval         8723 non-null   object 
 11  Etat arc                   8723 non-null   object 
 12  Date debut dispo data      8723 non-null   object 
 13  Date fin dispo data        8723 non-null   objec

In [10]:
df_final = df_final.copy()
df_final['Date et heure de comptage'] = pd.to_datetime(df_final['Date et heure de comptage'], errors='coerce')
df_final = df_final.sort_values('Date et heure de comptage').reset_index(drop=True)

for col in ['est_vacances', 'est_ferie', 'est_avant_ferie', 'est_pieton']:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

features = [
    'Température', 'est_vacances', 'duree prec (en min)',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)', 'jour_semaine', 'mois',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]
target = 'Débit horaire'

mask_known   = df_final[target].notna()
mask_missing = df_final[target].isna()

X_known = df_final.loc[mask_known, features].copy()
y_known = df_final.loc[mask_known, target].astype(float)
X_missing = df_final.loc[mask_missing, features].copy()

numeric_features = [
    'Température', 'est_vacances',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)', 'duree prec (en min)', 'mois',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]
categorical_features = ['jour_semaine']

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)         

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features),
    ],
    remainder='drop'
)

model = HistGradientBoostingRegressor(
    loss='absolute_error',   
    max_depth=6,
    max_iter=400,
    early_stopping=False,
    random_state=42
)

pipe = Pipeline(steps=[('prep', preprocess), ('model', model)])

# ---------- 3) Split chronologique ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_known, y_known, test_size=0.2, shuffle=False
)

# ---------- 4) Pondérations (férié / veille / piéton) ----------
W_FERIE   = 3.0
W_AVANT   = 1.5
W_PIETON  = 10
POST_SCALE_FERIE = 1.0  # laissez 1.0 si vous ne souhaitez pas corriger les prédictions les jours fériés

def make_weights(X_frame):
    w = np.ones(len(X_frame), dtype=float)
    is_ferie  = X_frame['est_ferie'].fillna(0).astype(int).to_numpy()
    is_avant  = X_frame['est_avant_ferie'].fillna(0).astype(int).to_numpy()
    is_pieton = X_frame['est_pieton'].fillna(0).astype(int).to_numpy()

    w[is_ferie == 1]  = W_FERIE
    w[is_avant == 1]  = np.maximum(w[is_avant == 1], W_AVANT)
    w[is_pieton == 1] = W_PIETON

    # Option : normalisation pour garder une échelle de perte comparable
    #w *= (len(w) / w.sum())
    return w

w_train = make_weights(X_train)

# ---------- 5) Entraînement ----------
pipe.fit(X_train, y_train, model__sample_weight=w_train)

# ---------- 6) Prédiction + post-ajustement éventuel ----------
y_pred = pipe.predict(X_test)

mask_ferie_test = X_test['est_ferie'].fillna(0).astype(int).to_numpy() == 1
y_pred[mask_ferie_test] *= POST_SCALE_FERIE

# ---------- 7) Évaluation ----------
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

# Diagnostics par sous-régimes
is_pieton_test = X_test['est_pieton'].fillna(0).astype(int) == 1
print(f"Part d'observations piéton (test) : {is_pieton_test.mean():.1%}")
if is_pieton_test.any():
    mae_pieton = mean_absolute_error(y_test[is_pieton_test], y_pred[is_pieton_test])
    print(f"MAE (jours piéton) : {mae_pieton:.2f} (n={is_pieton_test.sum()})")
    mae_non_pieton = mean_absolute_error(y_test[~is_pieton_test], y_pred[~is_pieton_test])
    print(f"MAE (jours non piéton) : {mae_non_pieton:.2f} (n={(~is_pieton_test).sum()})")

# ---------- 8) Imputation des valeurs manquantes ----------
if mask_missing.any():
    df_final.loc[mask_missing, 'Débit_prédit'] = pipe.predict(X_missing)
    print(f"Imputation réalisée pour {mask_missing.sum()} lignes (colonne 'Débit_prédit').")


R² : 0.878
MAE : 56.98
RMSE : 71.05
Part d'observations piéton (test) : 0.0%
Imputation réalisée pour 7343 lignes (colonne 'Débit_prédit').


In [11]:
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np
import plotly.express as px

# 1) Importance par permutation sur le pipeline complet (prétraitements inclus)
perm = permutation_importance(
    estimator=pipe,
    X=X_test,
    y=y_test,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  # cohérent avec votre RMSE
)

imp_df = (
    pd.DataFrame({
        'feature': features,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

# 2) Bar chart Plotly (top 20)
topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


                     feature  importance_mean  importance_std
3                  heure_sin       182.357746        8.775742
4                  heure_cos        81.903691        5.417367
5                   jour_sin        33.783658        3.306465
6                   jour_cos         7.558701        1.884148
10              jour_semaine         2.638216        0.844269
1               est_vacances         2.557027        0.774372
7                   mois_sin         2.020038        1.196654
0                Température         0.921698        0.717121
12                 est_ferie         0.439742        0.217304
13           est_avant_ferie         0.126326        0.056819
2        duree prec (en min)         0.060704        0.205518
8                   mois_cos         0.000000        0.000000
11                      mois         0.000000        0.000000
15                est_pieton         0.000000        0.000000
9   force moyenne vent (m/s)        -0.051572        0.406469
14   ens

In [12]:
time_index = df_final.loc[X_test.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_pred,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_test,
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()